### **Lấy và làm sạch Dữ liệu AQI và Weather từ AQI**

![Ảnh](./images/pam1_JLLG.png)

#### **I. Import các thư viện**

In [2]:
import requests
import json
import pandas as pd
import numpy as np
import datetime
from datetime import datetime, timedelta
from time import sleep
%matplotlib inline      

#### **II. Lấy dữ liệu từ API và tạo Dataframe từ data lấy được**
Trong dự án này, dữ liệu chất lượng không khí được thu thập thông qua [Weatherbit API](https://www.weatherbit.io/), một nền tảng tổng hợp dữ liệu khí tượng – môi trường toàn cầu.Weatherbit không trực tiếp đặt cảm biến tại từng vị trí địa lý, mà tổng hợp dữ liệu từ nhiều nguồn khác nhau, bao gồm:
- Các trạm quan trắc mặt đất (ground-based stations),

- Dữ liệu vệ tinh khí tượng (NASA MODIS, TROPOMI, MOPITT, v.v.),

- Và các mô hình khí tượng – hóa học (numerical models) như WRF-Chem, CAMS, GEOS-Chem.

Dữ liệu sau khi được thu thập sẽ được hiệu chỉnh sai số, nội suy không gian và chuẩn hóa theo từng ô lưới (grid) có kích thước khoảng 10 km × 10 km.
Điều này có nghĩa là các khu vực nằm trong cùng một ô lưới (ví dụ các quận nội thành Hà Nội nằm gần nhau) sẽ nhận được cùng một giá trị AQI và nồng độ các chất ô nhiễm (PM₂.₅, PM₁₀, CO, NO₂, SO₂, O₃).

Vì vậy, để tránh hiện tượng dữ liệu bị trùng lặp (nhân bản) giữa các quận lân cận, dự án lựa chọn tọa độ trung tâm của quận Hoàn Kiếm (21.0285°N, 105.8542°E) làm điểm đại diện cho khu vực nội thành Hà Nội. Hoàn Kiếm là khu vực trung tâm thủ đô, có mật độ dân cư cao, nhiều hoạt động giao thông và thương mại, do đó phản ánh tương đối chính xác chất lượng không khí trung bình của toàn khu vực đô thị Hà Nội.


In [2]:
# # API_KEY = "d0abdba555a24c308b658ff1a9af5267"
# API_KEY = "edb2db4cc2b84bec8a09cc37173ca0cc"
# LAT, LON = 21.0285, 105.8542
#
# start_date = datetime(2025, 2, 1)
# end_date = datetime(2025, 10, 30, 0)
#
# urls_air = []
# urls_wea = []
# current = start_date
# while current < end_date:
#     next_month = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
#     start_str = current.strftime('%Y-%m-%d')
#     end_str = next_month.strftime('%Y-%m-%d')
#
#     url_air = f"https://api.weatherbit.io/v2.0/history/airquality?lat={LAT}&lon={LON}&start_date={start_str}&end_date={end_str}&tz=local&key={API_KEY}"
#     url_wea = f"https://api.weatherbit.io/v2.0/history/hourly?lat={LAT}&lon={LON}&start_date={start_str}&end_date={end_str}&tz=local&key={API_KEY}"
#     urls_air.append(url_air)
#     urls_wea.append(url_wea)
#     current = next_month
#
# print(f" Tạo{len(urls_air)} URLs ({urls_air[0]} → {urls_air[-1]})")
# print(f" Tạo{len(urls_wea)} URLs ({urls_wea[0]} → {urls_wea[-1]})")
#

 Tạo9 URLs (https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-02-01&end_date=2025-03-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc → https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-10-01&end_date=2025-11-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc)
 Tạo9 URLs (https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-02-01&end_date=2025-03-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc → https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-10-01&end_date=2025-11-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc)


##### **2.1 AIR QUALITY**

In [3]:
# results_air = []
# for i, url in enumerate(urls_air):
#     print(f'Lấy dữ liệu từ URL {i}/{len(urls_air)} : {url}')
#     try:
#         renponse = requests.get(url, timeout=30)
#         renponse.raise_for_status()
#         data = json.loads(renponse.text)
#         results_air.append(data)
#         sleep(1.2)
#
#     except Exception as e:
#         print(f"Lỗi khi lấy dữ liệu {i} : {e}")
#
# print(f"\n Hoàn tất tải {len(results_air)} ")

Lấy dữ liệu từ URL 0/9 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-02-01&end_date=2025-03-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 1/9 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-03-01&end_date=2025-04-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 2/9 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-04-01&end_date=2025-05-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 3/9 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-05-01&end_date=2025-06-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 4/9 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-06-01&end_date=2025-07-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 5/9 : https://api.weatherbit.io/v2.0/hist

In [4]:
# results_air[0]['city_name']

'Hoàn Kiếm'

In [5]:


# results_air[0]['data'][0]

{'aqi': 131,
 'co': 10,
 'datetime': '2025-02-28:17',
 'no2': 61,
 'o3': 54.7,
 'pm10': 64,
 'pm25': 47,
 'so2': 39.7,
 'timestamp_local': '2025-03-01T00:00:00',
 'timestamp_utc': '2025-02-28T17:00:00',
 'ts': 1740762000}

In [6]:
# joined_data = []
# for res in results_air:
#     if 'data' in res:
#         joined_data.extend(res['data'])
#
# joined_results = {
#     'city_name': results_air[0]['city_name'],
#     'country_code': results_air[0]['country_code'],
#     'lat': results_air[0]['lat'],
#     'lon': results_air[0]['lon'],
#     'timezone': results_air[0]['timezone'],
#     'data': joined_data
# }



In [7]:
# joined_results['data'][0]

{'aqi': 131,
 'co': 10,
 'datetime': '2025-02-28:17',
 'no2': 61,
 'o3': 54.7,
 'pm10': 64,
 'pm25': 47,
 'so2': 39.7,
 'timestamp_local': '2025-03-01T00:00:00',
 'timestamp_utc': '2025-02-28T17:00:00',
 'ts': 1740762000}

In [8]:
# joined_results['data'][-1]

{'aqi': 65,
 'co': 95.7,
 'datetime': '2025-09-30:17',
 'no2': 19.7,
 'o3': 25.3,
 'pm10': 23.8,
 'pm25': 19,
 'so2': 43,
 'timestamp_local': '2025-10-01T00:00:00',
 'timestamp_utc': '2025-09-30T17:00:00',
 'ts': 1759251600}

In [9]:

# df = pd.DataFrame(joined_results)
#
# df.columns = ['City', 'Country code', 'Lat', 'Lon', 'timezone', 'Data']
# df[['AQI', 'CO', 'Date Time', 'NO2', 'O3', 'PM10', 'PM25', 'SO2', 'Local Time', 'UTC Time', 'TS']] = pd.DataFrame(df['Data'].tolist())
# df.drop(columns=['Data', 'Lat', 'Lon', 'TS', 'Date Time'], inplace=True)
# df = df.drop_duplicates()
# df = df.sort_values(by='Local Time')
# df['Local Time'] = pd.to_datetime(df['Local Time'])
# df.set_index('Local Time', inplace=True)
# df.head()

,City,Country code,timezone,AQI,CO,NO2,O3,PM10,PM25,SO2,UTC Time
Local Time,,,,,,,,,,,
2025-02-01 00:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,74,10.0,22.0,46.7,27.0,23.0,40.3,2025-01-31T17:00:00
2025-02-01 01:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,137,111.5,17.0,42.0,61.3,49.0,39.0,2025-01-31T18:00:00
2025-02-01 02:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,89,10.0,18.0,43.7,30.0,30.0,34.7,2025-01-31T19:00:00
2025-02-01 03:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,98,10.0,24.0,45.3,36.0,34.0,30.3,2025-01-31T20:00:00
2025-02-01 04:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,99,10.0,24.0,47.0,36.0,34.5,26.0,2025-01-31T21:00:00


In [3]:
# df.to_csv('air_quality_data.csv', index=True)

NameError: name 'df' is not defined

In [4]:
df = pd.read_csv('air_quality_data.csv')
df.shape

(6541, 12)

##### **2.2 WEATHER**

In [13]:
# results_wea = []
# for i, url in enumerate(urls_wea):
#     print(f'Lấy dữ liệu từ URL {i}/{len(urls_wea)} : {url}')
#     try:
#         renponse = requests.get(url, timeout=30)
#         renponse.raise_for_status()
#         data = json.loads(renponse.text)
#         results_wea.append(data)
#         sleep(1.2)
#
#     except Exception as e:
#         print(f"Lỗi khi lấy dữ liệu {i} : {e}")
#
# print(f"\n Hoàn tất tải {len(results_wea)} ")

Lấy dữ liệu từ URL 0/9 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-02-01&end_date=2025-03-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 1/9 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-03-01&end_date=2025-04-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 2/9 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-04-01&end_date=2025-05-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 3/9 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-05-01&end_date=2025-06-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 4/9 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-06-01&end_date=2025-07-01&tz=local&key=edb2db4cc2b84bec8a09cc37173ca0cc
Lấy dữ liệu từ URL 5/9 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.02

In [14]:
# joined_data = []
# for res in results_wea:
#     if 'data' in res:
#         joined_data.extend(res['data'])
#
# joined_results_weather = {
#     'city_name': results_wea[0]['city_name'],
#     'country_code': results_wea[0]['country_code'],
#     'lat': results_wea[0]['lat'],
#     'lon': results_wea[0]['lon'],
#     'timezone': results_wea[0]['timezone'],
#     'data': joined_data
# }

In [15]:

# df_weather = pd.DataFrame(joined_results_weather)


# df_weather.columns = ['City', 'Country code', 'Lat', 'Lon', 'timezone', 'Data']
#
# d = pd.json_normalize(df_weather.pop('Data'))
#
# keep = d[["clouds","precip","pres","rh","temp","uv","wind_spd","timestamp_local","timestamp_utc"]].rename(columns={
#     "clouds":"Clouds",
#     "precip":"Precipitation",
#     "pres":"Pressure",
#     "rh":"Relative Humidity",
#     "temp":"Temperature",
#     "uv":"UV Index",
#     "wind_spd":"Wind Speed",
#     "timestamp_local":"Local Time",
#     "timestamp_utc":"UTC Time"
# })
# df_weather = pd.concat([df_weather[['City', 'Country code', 'timezone']], keep], axis=1)
#
# df_weather.head()

,City,Country code,timezone,Clouds,Precipitation,Pressure,Relative Humidity,Temperature,UV Index,Wind Speed,Local Time,UTC Time
0,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,80.0,0.0,1011.0,89.0,19.2,0.0,1.0,2025-02-01T00:00:00,2025-01-31T17:00:00
1,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,50.0,0.0,1011.0,91.0,19.0,0.0,1.0,2025-02-01T01:00:00,2025-01-31T18:00:00
2,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,54.0,0.0,1011.0,91.0,18.9,0.0,1.0,2025-02-01T02:00:00,2025-01-31T19:00:00
3,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,58.0,0.0,1011.0,91.0,18.9,0.0,1.0,2025-02-01T03:00:00,2025-01-31T20:00:00
4,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,62.0,0.0,1011.0,91.0,18.8,0.0,1.0,2025-02-01T04:00:00,2025-01-31T21:00:00


In [16]:
# df_weather.to_csv('weather_data.csv', index=False)

In [7]:
df_weather = pd.read_csv('weather_data.csv')
# df_weather.info()


#### **IV. Hợp nhất hai khung dữ liệu và sắp xếp dữ liệu**

In [8]:

merged_df = pd.merge(df, df_weather, left_index=True, right_index=True)

merged_df.drop(columns=['City_y', 'Country code_y', 'timezone_y', 'UTC Time_y'], inplace=True)

utc_time_column = merged_df.pop('UTC Time_x')
merged_df.insert(0, 'UTC Time', utc_time_column)

merged_df = merged_df.rename(columns={'City_x': 'City', 'Country code_x': 'Country Code', 'timezone_x':'Timezone'})
merged_df

,UTC Time,Local Time_x,City,Country Code,Timezone,AQI,CO,NO2,O3,PM10,PM25,SO2,Clouds,Precipitation,Pressure,Relative Humidity,Temperature,UV Index,Wind Speed,Local Time_y
0,2025-01-31T17:00:00,2025-02-01 00:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,74,10.0,22.0,46.7,27.0,23.00,40.3,80.0,0.00,1011.0,89.0,19.2,0.0,1.00,2025-02-01T00:00:00
1,2025-01-31T18:00:00,2025-02-01 01:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,137,111.5,17.0,42.0,61.3,49.00,39.0,50.0,0.00,1011.0,91.0,19.0,0.0,1.00,2025-02-01T01:00:00
2,2025-01-31T19:00:00,2025-02-01 02:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,89,10.0,18.0,43.7,30.0,30.00,34.7,54.0,0.00,1011.0,91.0,18.9,0.0,1.00,2025-02-01T02:00:00
3,2025-01-31T20:00:00,2025-02-01 03:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,98,10.0,24.0,45.3,36.0,34.00,30.3,58.0,0.00,1011.0,91.0,18.9,0.0,1.00,2025-02-01T03:00:00
4,2025-01-31T21:00:00,2025-02-01 04:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,99,10.0,24.0,47.0,36.0,34.50,26.0,62.0,0.00,1011.0,91.0,18.8,0.0,1.00,2025-02-01T04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6536,2025-10-30T23:00:00,2025-10-31 06:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,68,2074.5,33.7,48.7,25.7,20.25,9.7,85.0,2.50,1016.0,93.0,21.7,1.2,2.48,2025-10-31T08:00:00
6537,2025-10-31T00:00:00,2025-10-31 07:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,121,151.2,12.0,49.0,43.0,43.00,70.0,99.0,0.25,1017.0,92.0,22.1,1.5,2.36,2025-10-31T09:00:00
6538,2025-10-31T01:00:00,2025-10-31 08:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,61,2554.3,52.5,42.4,27.0,17.20,15.5,100.0,0.00,1017.0,93.0,21.9,2.0,2.76,2025-10-31T10:00:00
6539,2025-10-31T02:00:00,2025-10-31 09:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,67,2111.5,32.7,31.0,24.3,20.00,10.7,NaN,0.00,NaN,NaN,NaN,NaN,NaN,2025-10-31T11:00:00


In [9]:

merged_df.shape

(6541, 20)

In [11]:
merged_df.rename(columns={'Local Time_x': 'Local Time'}, inplace=True)
merged_df = merged_df[['Local Time', 'UTC Time', 'City', 'Country Code', 'Timezone', 'AQI', 'CO', 'NO2', 'O3', 'PM10', 'PM25', 'SO2',
                       'Clouds', 'Precipitation', 'Pressure', 'Relative Humidity', 'Temperature', 'UV Index', 'Wind Speed']]

merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6541 entries, 0 to 6540
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Local Time         6541 non-null   object 
 1   UTC Time           6541 non-null   object 
 2   City               6541 non-null   object 
 3   Country Code       6541 non-null   object 
 4   Timezone           6541 non-null   object 
 5   AQI                6541 non-null   int64  
 6   CO                 6541 non-null   float64
 7   NO2                6541 non-null   float64
 8   O3                 6541 non-null   float64
 9   PM10               6541 non-null   float64
 10  PM25               6541 non-null   float64
 11  SO2                6541 non-null   float64
 12  Clouds             6539 non-null   float64
 13  Precipitation      6541 non-null   float64
 14  Pressure           6539 non-null   float64
 15  Relative Humidity  6539 non-null   float64
 16  Temperature        6539 non-n

In [12]:

merged_df.to_csv('hanoi-aqi-weather-data-TEST.csv', index=False)